# 02. SCM 합성 과제와 Zero-shot 문맥 예측

## 목표
구조적 인과 모델로 서로 다른 표 분류 과제를 만들고, 새 과제마다 가중치를 학습하지 않는 문맥 기반 예측을 실험합니다.

In [ ]:
from math import exp, sqrt
from random import Random

def make_task(seed, rows=80):
    rng = Random(seed)
    # 과제마다 인과 함수의 계수를 바꿔 다양한 데이터 생성 과정을 만듭니다.
    weight_x = rng.uniform(-2.0, 2.0)
    weight_z = rng.uniform(-2.0, 2.0)
    bias = rng.uniform(-0.5, 0.5)
    data = []
    for _ in range(rows):
        x = rng.gauss(0, 1)
        z = 0.7 * x + rng.gauss(0, 0.8)
        score = weight_x * x + weight_z * z + bias + rng.gauss(0, 0.2)
        data.append(({"x": x, "z": z}, int(score > 0)))
    return data

tasks = [make_task(seed) for seed in range(5)]
print([sum(label for _, label in task) for task in tasks])

In [ ]:
def squared_distance(a, b):
    return sum((a[name] - b[name]) ** 2 for name in a)

def context_probability(context, row, neighbors=7):
    nearest = sorted(context, key=lambda item: squared_distance(item[0], row))[:neighbors]
    # 거리 가중 평균은 새 과제의 레이블 예시만 사용하며 전역 파라미터를 fit하지 않습니다.
    weights = [1.0 / (sqrt(squared_distance(features, row)) + 1e-6) for features, _ in nearest]
    return sum(weight * label for weight, (_, label) in zip(weights, nearest)) / sum(weights)

def evaluate_task(task, context_size=40):
    context, test = task[:context_size], task[context_size:]
    correct = 0
    for row, label in test:
        prediction = int(context_probability(context, row) >= 0.5)
        correct += prediction == label
    return correct / len(test)

for index, task in enumerate(tasks):
    print(f"task {index}: accuracy={evaluate_task(task):.3f}")

## 실험

- `context_size`를 10, 20, 40, 60으로 바꿔 예시 수 효과를 비교하세요.
- 노이즈 크기를 늘리고 성능 변화를 확인하세요.
- 비선형 항 `x * z`를 추가해 단순 거리 기반 방법의 한계를 관찰하세요.
- 합성 prior가 실제 과제 구조를 포함하지 않을 때 일반화가 왜 어려운지 설명하세요.